In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Calculation Without Data: NaCl, X-ray

This is the most minimal "calculation without data" example. It
defines an X-ray powder experiment for NaCl, **accepts the default
calculation range**, and plots the calculated pattern — no data file
and no manual range setup.

The default `data_range` is derived from the instrument (here the
X-ray wavelength), so a structure-only experiment is calculable
immediately.

## 🛠️ Import Library

In [2]:
import numpy as np

import easydiffraction as edi

## 📦 Define Project

In [3]:
project = edi.Project(name='nacl_simulation')

## 🧩 Define Structure

In [4]:
project.structures.create(name='nacl')

In [5]:
structure = project.structures['nacl']

In [6]:
structure.space_group.name_h_m = 'F m -3 m'
structure.space_group.coord_system_code = '1'

In [7]:
structure.cell.length_a = 5.62

In [8]:
structure.atom_sites.create(
    id='Na',
    type_symbol='Na',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    adp_iso=0.5,
)
structure.atom_sites.create(
    id='Cl',
    type_symbol='Cl',
    fract_x=0.5,
    fract_y=0.5,
    fract_z=0.5,
    adp_iso=0.5,
)

## 🔬 Define Experiment

### Create Experiment Without Data

In [9]:
project.experiments.create(
    name='sim',
    sample_form='powder',
    beam_mode='constant wavelength',
    radiation_probe='xray',
)

In [10]:
experiment = project.experiments['sim']

### Set Instrument

In [11]:
experiment.instrument.setup_wavelength = 1.5406

### Set Peak Profile

In [12]:
experiment.peak.broad_gauss_u = 0.1
experiment.peak.broad_gauss_v = -0.1
experiment.peak.broad_gauss_w = 0.1

### Set Linked Structures

In [13]:
experiment.linked_structures.create(structure_id='nacl', scale=1.0)

### Inspect the Default Calculation Range

No range is set explicitly: the default window (derived from the
wavelength) is used as-is.

In [14]:
print('min:', experiment.data_range.two_theta_min.value)
print('max:', experiment.data_range.two_theta_max.value)
print('inc:', experiment.data_range.two_theta_inc.value)

min: 8.835740516099717
max: 170.0
inc: 0.16132558506896924


### Set X-ray Polarization Optics

The polarization coefficient defaults to `0.0`. Setting it to a
nonzero value includes the Lorentz-polarization optics in the
calculated X-ray intensities.

In [15]:
project.analysis.calculate()
unpolarized_intensity = np.asarray(experiment.data.intensity_calc).copy()

In [16]:
experiment.instrument.setup_polarization_coefficient = 0.5
experiment.instrument.setup_monochromator_twotheta = 26.5650511771

In [17]:
project.analysis.calculate()
polarized_intensity = np.asarray(experiment.data.intensity_calc).copy()

In [18]:
print('max intensity change:', np.max(np.abs(polarized_intensity - unpolarized_intensity)))

max intensity change: 236202.50313200592


## 🚀 Perform Calculation

### Display Pattern

In [19]:
project.display.pattern(expt_name='sim')

## 💾 Save Project

In [20]:
project.save_as(dir_path='projects/simulate-nacl-xray')

Saving project 📦 'nacl_simulation' to '../../../projects/simulate-nacl-xray'


├── 📄 project.edi


├── 📁 structures/


│   └── 📄 nacl.edi


├── 📁 experiments/


│   └── 📄 sim.edi


├── 📁 analysis/


│   └── 📄 analysis.edi


└── 📁 reports/


    └── 📄 nacl_simulation.html
